# Calculate NDVI for Platero L1C Image

This notebook calculates the Normalized Difference Vegetation Index (NDVI) for a **Platero L1C** image. Platero data is provided as separate per-band COG assets (R, G, B, NIR, RE1, RE2, RE3, PAN, etc.).

## What is NDVI?

NDVI (Normalized Difference Vegetation Index) uses the difference between near-infrared (NIR) and red light reflectance to assess vegetation health and density. Values range from -1 to 1:

- **Values close to 1**: Dense, healthy vegetation
- **Values around 0**: Bare soil or sparse vegetation
- **Values close to -1**: Water or other non-vegetated surfaces

## Parameters

- **Collection**: `platero-l1C-cogs` (or your STAC collection)
- **STAC Item**: Link to the Platero L1C STAC item
- **AOI (Area of Interest)**: Optional GeoJSON; if not provided, a 1000×1000 pixel window from the center is used

## Workflow

1. Load the STAC item
2. Access the **R** (red) and **NIR** (near-infrared) band assets
3. Read band data and apply scale/offset from STAC for physical values
4. Calculate NDVI: (NIR − Red) / (NIR + Red)
5. Visualize the results

## Step 1: Import Required Libraries

In [ ]:
import pystac
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import json
import warnings


warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

In [ ]:
%pip install python-dotenv
from dotenv import load_dotenv
import os

If you wish to load data from non-open datasets you need to provide a token

In [ ]:
load_dotenv()
token = os.getenv("token")
# So rasterio/GDAL send the token when opening COG URLs (avoids 401 on asset hrefs)
if token:
    os.environ["GDAL_HTTP_HEADERS"] = f"Authorization: Bearer {token}"

Set variables

In [ ]:
stac_item_url = "https://eodatahub.org.uk/api/catalogue/stac/catalogs/user/catalogs/samples-opencosmos/catalogs/commercial-data/catalogs/opencosmos/collections/platero-l1C-cogs/items/platero_l1c_000002275_20251009163434_20251009163437_9d217923"
stac_collection_name = "platero-l1C-cogs"
aoi_param = """{{AOI}}""".strip()

In [ ]:
stac_io = pystac.StacIO.default()
stac_io.headers = {"Authorization": f"Bearer {token}"}


def load_stac_item(stac_item_url, token):
    # Use token for authenticated STAC access
    try:
        # Load the STAC item with auth
        item = pystac.Item.from_file(stac_item_url, stac_io=stac_io)
        print(f"Successfully loaded STAC item: {item.id}")
        return item

    except Exception as e:
        print(f"Error loading STAC item: {e}")
        raise


item = load_stac_item(stac_item_url, token)

## Step 2: Determine Area of Interest (AOI)

If an AOI is provided (GeoJSON), the raster will be clipped to that geometry. Otherwise, a 1000×1000 pixel window from the center is used.

In [ ]:
aoi_param = ""

DEFAULT_WINDOW_SIZE = 1000
aoi_geometry = None
clip_window = None
use_windowed_read = False

if aoi_param and aoi_param.strip().lower() not in ("", "none", "null"):
    try:
        aoi_data = json.loads(aoi_param)
        if aoi_data.get("type") == "FeatureCollection":
            if aoi_data.get("features") and len(aoi_data["features"]) > 0:
                aoi_geometry = aoi_data["features"][0].get("geometry")
        elif aoi_data.get("type") == "Feature":
            aoi_geometry = aoi_data.get("geometry")
        elif aoi_data.get("type") in ["Polygon", "MultiPolygon", "Point", "LineString"]:
            aoi_geometry = aoi_data
        else:
            raise ValueError(f"Unsupported GeoJSON type: {aoi_data.get('type')}")
        if aoi_geometry and aoi_geometry.get("type"):
            print(f"AOI provided: {aoi_geometry['type']} geometry")
        else:
            raise ValueError("Could not extract geometry from GeoJSON")
    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Warning: Could not parse AOI: {e}. Using default window.")
        aoi_geometry = None
else:
    print("No AOI provided, using default 1000×1000 pixel window")

if aoi_geometry is None:
    use_windowed_read = True
    print(
        f"Will extract {DEFAULT_WINDOW_SIZE}×{DEFAULT_WINDOW_SIZE} pixel window from center"
    )

## Step 3: Access Platero R and NIR Bands

Platero L1C items expose bands as separate assets: **R** (red, ~665 nm), **NIR** (near-infrared, ~837 nm). We also read each band's `scale` and `offset` from the STAC asset metadata so NDVI uses physical values.

In [ ]:
def get_scale_offset(asset):
    """Read scale and offset from Platero asset bands metadata."""
    extra = getattr(asset, "extra", {}) or {}
    bands = extra.get("bands", [])
    if bands:
        b = bands[0]
        return float(b.get("scale", 1.0)), float(b.get("offset", 0.0))
    return 1.0, 0.0


try:
    if "R" not in item.assets:
        raise ValueError(
            "This STAC item has no 'R' (red) asset. Available: "
            + ", ".join(item.assets.keys())
        )
    if "NIR" not in item.assets:
        raise ValueError(
            "This STAC item has no 'NIR' asset. Available: "
            + ", ".join(item.assets.keys())
        )

    red_band_asset = item.assets["R"]
    nir_band_asset = item.assets["NIR"]

    red_scale, red_offset = get_scale_offset(red_band_asset)
    nir_scale, nir_offset = get_scale_offset(nir_band_asset)

    print("Platero band assets found:")
    print(f"  R (red):   {red_band_asset.href}")
    print(f"  NIR:       {nir_band_asset.href}")
    print(f"  R scale={red_scale}, offset={red_offset}")
    print(f"  NIR scale={nir_scale}, offset={nir_offset}")
except Exception as e:
    print(f"Error accessing bands: {e}")
    raise

## Step 4: Read Band Data and Apply Scale/Offset

Read the R and NIR COGs (optionally clipped to AOI or a center window), then convert to physical values using each band's scale and offset.

In [ ]:
try:
    try:
        _ = aoi_geometry
    except NameError:
        aoi_geometry = None
        clip_window = None
        use_windowed_read = False

    with rasterio.open(red_band_asset.href) as red_src:
        if aoi_geometry is not None:
            red_data, red_transform = mask(red_src, [aoi_geometry], crop=True)
            red_data = red_data[0]
            red_profile = red_src.profile.copy()
            red_profile.update(
                {
                    "height": red_data.shape[0],
                    "width": red_data.shape[1],
                    "transform": red_transform,
                    "count": 1,
                }
            )
        elif use_windowed_read:
            height, width = red_src.height, red_src.width
            if height < DEFAULT_WINDOW_SIZE or width < DEFAULT_WINDOW_SIZE:
                clip_window = None
                red_data = red_src.read(1)
            else:
                row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                clip_window = Window(
                    col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                )
                red_data = red_src.read(1, window=clip_window)
            red_profile = red_src.profile.copy()
            if clip_window:
                red_profile.update(
                    {
                        "height": clip_window.height,
                        "width": clip_window.width,
                        "transform": rasterio.windows.transform(
                            clip_window, red_src.transform
                        ),
                        "count": 1,
                    }
                )
            else:
                red_profile.update(count=1)
        else:
            red_data = red_src.read(1)
            red_profile = red_src.profile.copy()
            red_profile.update(count=1)
        red_crs = red_src.crs

    with rasterio.open(nir_band_asset.href) as nir_src:
        if aoi_geometry is not None:
            nir_data, nir_transform = mask(nir_src, [aoi_geometry], crop=True)
            nir_data = nir_data[0]
        elif use_windowed_read and clip_window is not None:
            nir_data = nir_src.read(1, window=clip_window)
        else:
            nir_data = nir_src.read(1)

    red_data = red_data.astype(np.float32) * red_scale + red_offset
    nir_data = nir_data.astype(np.float32) * nir_scale + nir_offset

    if red_data.shape != nir_data.shape:
        raise ValueError(
            f"Band shapes do not match: R {red_data.shape} vs NIR {nir_data.shape}"
        )

    print(f"R band shape: {red_data.shape}, dtype: {red_data.dtype}")
    print(f"NIR band shape: {nir_data.shape}")
    print(f"CRS: {red_crs}")
    print(f"R range: [{np.nanmin(red_data):.4f}, {np.nanmax(red_data):.4f}]")
    print(f"NIR range: [{np.nanmin(nir_data):.4f}, {np.nanmax(nir_data):.4f}]")
except Exception as e:
    print(f"Error reading band data: {e}")
    raise

## Step 5: Calculate NDVI

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

In [ ]:
denominator = nir_data + red_data
valid_mask = denominator != 0
ndvi = np.full_like(red_data, np.nan, dtype=np.float32)
ndvi[valid_mask] = (nir_data[valid_mask] - red_data[valid_mask]) / denominator[
    valid_mask
]
ndvi = np.clip(ndvi, -1.0, 1.0)

print("NDVI calculation complete!")
print(
    f"NDVI min: {np.nanmin(ndvi):.4f}, max: {np.nanmax(ndvi):.4f}, mean: {np.nanmean(ndvi):.4f}"
)
print(f"Valid pixels: {np.sum(~np.isnan(ndvi)):,} of {ndvi.size:,}")

## Step 6: Visualize Results

In [ ]:
colors = ["#000080", "#0066CC", "#CCCCCC", "#FFFF00", "#00FF00", "#008000"]
cmap = LinearSegmentedColormap.from_list("ndvi", colors, N=256)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmax_red = (
    np.percentile(red_data[~np.isnan(red_data) & (red_data > 0)], 98)
    if np.any(red_data > 0)
    else np.nanmax(red_data)
)
vmax_nir = (
    np.percentile(nir_data[~np.isnan(nir_data) & (nir_data > 0)], 98)
    if np.any(nir_data > 0)
    else np.nanmax(nir_data)
)

axes[0].imshow(red_data, cmap="Reds", vmin=0, vmax=vmax_red)
axes[0].set_title("Red Band (R)", fontsize=14, fontweight="bold")
axes[0].axis("off")
plt.colorbar(axes[0].images[0], ax=axes[0], fraction=0.046, pad=0.04, label="Radiance")

axes[1].imshow(nir_data, cmap="YlGn", vmin=0, vmax=vmax_nir)
axes[1].set_title("NIR Band", fontsize=14, fontweight="bold")
axes[1].axis("off")
plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046, pad=0.04, label="Radiance")

im3 = axes[2].imshow(ndvi, cmap=cmap, vmin=-1, vmax=1)
axes[2].set_title("NDVI", fontsize=14, fontweight="bold")
axes[2].axis("off")
cbar = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, label="NDVI")
cbar.set_ticks([-1, -0.5, 0, 0.3, 0.6, 1])
cbar.set_ticklabels(["Water", "Bare Soil", "Sparse", "Moderate", "Dense", "Very Dense"])

plt.suptitle(f"NDVI — Platero L1C — {item.id}", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print("Visualization complete.")

## Summary

This notebook loaded a Platero L1C STAC item, read the **R** and **NIR** per-band COG assets, applied scale/offset from STAC metadata, and computed NDVI. Change `stac_item_url` (and optionally set `aoi_param` to a GeoJSON string) to run the same workflow on other Platero scenes.